# 02 — Pipeline de Preprocesamiento e Ingeniería de Características
## Cross-Selling Recommender | Proyecto Final Henry

**Objetivo:** Ejecutar el pipeline ETL completo sobre Instacart, documentar el tratamiento de calidad de datos y generar los artefactos de features necesarios para cada modelo.

**Entrada:** 6 CSV en `data/raw/`

**Salida:**
- `data/processed/master_prior.parquet` — DataFrame maestro completo (para CF/ALS)
- `data/processed/df_apriori.parquet` — Muestra 200K órdenes (para FP-Growth)
- `data/processed/purchase_frequency.parquet` — Señal usuario-producto para ALS

**Pasos:**
1. Carga con dtypes optimizados (RAM: ~4 GB → ~1.8 GB)
2. Calidad de datos: valores faltantes, duplicados, outliers, desbalance
3. Limpieza: deduplicación + filtro cestas pequeñas
4. Feature engineering: features temporales + frecuencia de compra
5. Muestreo estratificado para Apriori (200K órdenes)
6. Guardado de artefactos

In [ ]:
# ── 1. Configuración ─────────────────────────────────────────────────────────
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

PROJECT_ROOT = Path().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import (
    load_config,
    load_instacart_tables,
    load_train_split,
    build_master_dataframe,
)
from src.data.preprocessor import (
    run_cleaning_pipeline,
    compute_purchase_frequency,
    get_basket_size_stats,
)
from src.features.builder import build_basket_matrix

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR   = PROJECT_ROOT / "outputs" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")

---
## 2. Carga del Dataset

Instacart tiene 6 archivos. Los cargamos con dtypes optimizados para reducir RAM:

| Dtype elegido | Columnas | Reducción vs default |
|---|---|---|
| `int32` | order_id, user_id, product_id | −50% vs int64 |
| `uint8` | order_dow, order_hour_of_day, reordered | −75% vs int64 |
| `float32` | days_since_prior_order | −50% vs float64 |
| `int8` | order_number, department_id | −75% vs int64 |

In [ ]:
# ── 2.1 Cargar 6 tablas con dtypes optimizados ────────────────────────────────
cfg = load_config()

orders, op_prior, products, aisles, departments = load_instacart_tables(cfg)
df_train_split = load_train_split(cfg)   # evaluación: última orden por usuario

print("\nTablas cargadas:")
for name, df_t in [
    ("orders",               orders),
    ("order_products_prior", op_prior),
    ("order_products_train", df_train_split),
    ("products",             products),
    ("aisles",               aisles),
    ("departments",          departments),
]:
    mem = df_t.memory_usage(deep=True).sum() / 1e6
    print(f"  {name:<30} {len(df_t):>12,} filas  |  {mem:>6.1f} MB")

product_names = products.set_index("product_id")["product_name"].to_dict()

In [ ]:
# ── 2.2 Construir DataFrame maestro (JOIN progresivo de 5 tablas) ─────────────
# Solo split 'prior' (histórico de compras) → base de entrenamiento de los modelos
df_master = build_master_dataframe(
    orders=orders,
    order_products=op_prior,
    products=products,
    aisles=aisles,
    departments=departments,
    eval_set="prior",
)

print(f"\nDataFrame maestro: {df_master.shape}")
print(f"Columnas: {list(df_master.columns)}")
df_master.head(3)

---
## 3. Calidad de Datos

### 3.1 Valores Faltantes

In [ ]:
# ── 3.1 Análisis de valores faltantes ────────────────────────────────────────
missing     = df_master.isnull().sum()
missing_pct = (missing / len(df_master) * 100).round(3)

df_missing = pd.DataFrame({
    "Nulos": missing,
    "% del total": missing_pct,
    "Dtype": df_master.dtypes,
}).sort_values("Nulos", ascending=False)

print("Valores faltantes por columna:")
display(df_missing)

print()
print("📌 DECISIÓN — days_since_prior_order:")
print("   Los NaN son primeras órdenes de usuarios (no tienen orden previa).")
print("   Son información válida, no errores. Estrategia: conservar como NaN")
print("   y convertir a categoría 'primera_orden' en la feature days_bucket.")
print("   El resto del dataset no tiene nulos — no se requiere imputación.")

In [ ]:
# ── 3.2 Visualización de nulos + distribución de days_since_prior_order ───────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Calidad de datos — Valores faltantes", fontsize=12, fontweight="bold")

# Barras de % nulos
missing_nz = missing_pct[missing_pct > 0]
if len(missing_nz) > 0:
    axes[0].barh(missing_nz.index, missing_nz.values,
                 color="#d9534f", alpha=0.85, edgecolor="white")
    for i, v in enumerate(missing_nz.values):
        axes[0].text(v + 0.1, i, f"{v:.1f}%", va="center", fontsize=9)
    axes[0].set_xlabel("% de valores nulos")
else:
    axes[0].text(0.5, 0.5, "Sin valores faltantes", ha="center",
                 va="center", fontsize=13, color="green")
axes[0].set_title("% Nulos por columna")

# Distribución de days_since_prior_order
dspo = orders["days_since_prior_order"].dropna()
axes[1].hist(dspo, bins=32, color="#4C72B0", alpha=0.85, edgecolor="white")
axes[1].axvline(dspo.median(), color="red", linestyle="--", linewidth=1.5,
                label=f"Mediana: {dspo.median():.0f} días")
axes[1].set_xlabel("Días desde compra previa")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución days_since_prior_order\n(NaN = primera orden — excluidos)")
axes[1].legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / "etl_01_missing_values.png", bbox_inches="tight")
plt.show()
print("Guardado: outputs/figures/etl_01_missing_values.png")

### 3.2 Duplicados y Outliers

In [ ]:
# ── 3.3 Duplicados ────────────────────────────────────────────────────────────
n_full_dups = df_master.duplicated().sum()
n_key_dups  = df_master.duplicated(subset=["order_id", "product_id"]).sum()

print(f"Duplicados exactos              : {n_full_dups:,}")
print(f"Duplicados por (order_id, prod) : {n_key_dups:,}")
if n_key_dups > 0:
    print(f"  ⚠️  Se eliminarán en el pipeline (keep='first').")
else:
    print("  ✅ Sin duplicados clave.")

In [ ]:
# ── 3.4 Outliers: tamaño de cestas ───────────────────────────────────────────
# Fence de 3×IQR (más permisivo que 1.5× — adecuado para conteos de compras)
basket_sizes = df_master.groupby("order_id")["product_id"].nunique()

q1, q3  = basket_sizes.quantile(0.25), basket_sizes.quantile(0.75)
iqr     = q3 - q1
fence   = q3 + 3 * iqr
n_out   = (basket_sizes > fence).sum()

print(f"Basket size — estadísticas:")
print(f"  Q1={q1:.0f}  Mediana={basket_sizes.median():.0f}  Q3={q3:.0f}  Máx={basket_sizes.max()}")
print(f"  Fence 3×IQR = {fence:.0f}")
print(f"  Órdenes outlier (> {fence:.0f} ítems): {n_out:,} ({n_out/len(basket_sizes)*100:.2f}%)")
print()
print("📌 DECISIÓN: No se eliminan. Las cestas grandes son usuarios genuinamente")
print("   activos (familias, compras de semana). Eliminarlas introduciría sesgo.")
print("   Sí se elimina min_basket_items < 2 (órdenes de 1 ítem no generan pares).")

In [ ]:
# ── 3.5 Desbalance: tasa de recompra ─────────────────────────────────────────
reorder_dist = df_master["reordered"].value_counts(normalize=True) * 100

print("Distribución de 'reordered' (variable más desbalanceada):")
for val, pct in reorder_dist.items():
    label = "Primera vez" if val == 0 else "Recompra"
    bar   = "█" * int(pct / 2)
    print(f"  {val} ({label:<12}): {pct:5.1f}%  {bar}")

print()
print("📌 DECISIÓN: No se aplica oversampling ni undersampling.")
print("   FP-Growth usa frecuencia absoluta de co-ocurrencia (insensible).")
print("   ALS modela preferencia implícita (frecuencia → confianza), no clasifica.")
print("   El desbalance es estructural del dominio — retail tiene mayoría de")
print("   primeras compras porque el catálogo tiene 50K productos.")

In [ ]:
# ── 3.6 Validación de rangos ──────────────────────────────────────────────────
checks = [
    ("order_dow (0–6)",            df_master["order_dow"].between(0, 6)),
    ("order_hour_of_day (0–23)",   df_master["order_hour_of_day"].between(0, 23)),
    ("reordered (0 o 1)",          df_master["reordered"].isin([0, 1])),
    ("add_to_cart_order (> 0)",    df_master["add_to_cart_order"] > 0),
]
print("Validación de rangos esperados:")
for name, mask in checks:
    ok  = mask.all()
    pct = mask.mean() * 100
    print(f"  {'✅' if ok else '❌'} {name:<35}  {pct:.4f}% válido")

---
## 4. Pipeline de Limpieza

`run_cleaning_pipeline()` aplica en orden:
1. Validación de esquema (columnas requeridas)
2. Eliminación de duplicados `(order_id, product_id)`
3. Filtro cestas < 2 ítems
4. Features temporales: `is_weekend`, `time_of_day`, `days_bucket`
5. Muestreo aleatorio de 200K órdenes para Apriori

Retorna dos DataFrames: **`df_full`** (para ALS) y **`df_apriori`** (para FP-Growth).

In [ ]:
# ── 4.1 Ejecutar pipeline ────────────────────────────────────────────────────
df_full, df_apriori = run_cleaning_pipeline(
    df=df_master,
    min_basket_items=2,
    add_temporal=True,
    apriori_sample=200_000,
    random_state=RANDOM_STATE,
)

print(f"\nResultados del pipeline:")
print(f"  df_master   (crudo)    : {len(df_master):>12,} filas | {df_master['order_id'].nunique():,} órdenes")
print(f"  df_full     (CF/ALS)   : {len(df_full):>12,} filas | {df_full['order_id'].nunique():,} órdenes")
print(f"  df_apriori  (FP-Growth): {len(df_apriori):>12,} filas | {df_apriori['order_id'].nunique():,} órdenes")
print(f"\n  Retención: {len(df_full)/len(df_master)*100:.1f}%")
nuevas = [c for c in df_full.columns if c not in df_master.columns]
print(f"  Features añadidas: {nuevas}")

---
## 5. Ingeniería de Características

In [ ]:
# ── 5.1 Frecuencia de compra usuario-producto (señal para ALS) ───────────────
# Calcula (user_id, product_id) → purchase_count + reorder_rate
# La confianza ALS = 1 + alpha × purchase_count (Hu et al., 2008)

df_freq = compute_purchase_frequency(df_full)

print(f"Pares usuario-producto únicos : {len(df_freq):,}")
print(f"Columnas: {list(df_freq.columns)}")
print(f"\nEstadísticas de purchase_count:")
display(df_freq[["purchase_count", "reorder_rate"]].describe().round(3))

In [ ]:
# ── 5.2 Estadísticas de basket size post-limpieza ────────────────────────────
print("Basket size post-limpieza — df_full (base ALS):")
display(get_basket_size_stats(df_full).to_frame("df_full").T.round(2))

print("\nBasket size post-limpieza — df_apriori (base FP-Growth):")
display(get_basket_size_stats(df_apriori).to_frame("df_apriori").T.round(2))

---
## 6. Visualización Post-Limpieza

In [ ]:
# ── 6.1 Comparativa antes/después + features nuevas ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Pipeline de preprocesamiento — resultados", fontsize=12, fontweight="bold")

# Volumen de filas por etapa
stages = ["Master crudo", "df_full (CF)", "df_apriori (AR)"]
n_rows = [len(df_master), len(df_full), len(df_apriori)]
bars   = axes[0].bar(stages, n_rows,
                     color=["#d9534f", "#5cb85c", "#5bc0de"],
                     alpha=0.85, edgecolor="white")
axes[0].set_title("Volumen de filas por etapa")
axes[0].set_ylabel("Nº de filas")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
for bar, val in zip(bars, n_rows):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f"{val/1e6:.1f}M", ha="center", fontsize=9)

# Basket size antes vs después
bs_raw  = df_master.groupby("order_id")["product_id"].nunique()
bs_full = df_full.groupby("order_id")["product_id"].nunique()
axes[1].hist(bs_raw.clip(upper=50),  bins=30, alpha=0.5, color="#d9534f", label="Crudo",  density=True)
axes[1].hist(bs_full.clip(upper=50), bins=30, alpha=0.5, color="#5cb85c", label="Limpio", density=True)
axes[1].set_xlabel("Ítems/orden (cap. 50)")
axes[1].set_title("Basket size: antes vs después")
axes[1].legend(fontsize=9)

# Features generadas
nuevas_feats = [c for c in df_full.columns if c not in df_master.columns]
axes[2].barh(nuevas_feats, [1]*len(nuevas_feats),
             color="#8B78BE", alpha=0.85)
axes[2].set_title("Features de ingeniería añadidas")
axes[2].get_xaxis().set_visible(False)
for i, feat in enumerate(nuevas_feats):
    axes[2].text(0.03, i, feat, va="center", fontsize=9, color="white", fontweight="bold")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "etl_02_pipeline_results.png", bbox_inches="tight")
plt.show()
print("Guardado: outputs/figures/etl_02_pipeline_results.png")

In [ ]:
# ── 6.2 Distribución de features de ingeniería ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Features temporales generadas por el pipeline", fontsize=12, fontweight="bold")

# is_weekend
wd = df_full["is_weekend"].value_counts().rename({0: "Semana", 1: "Fin de semana"})
axes[0].bar(wd.index, wd.values, color=["#4C72B0", "#DD8452"], alpha=0.85, edgecolor="white")
axes[0].set_title("is_weekend")
axes[0].set_ylabel("Órdenes")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

# time_of_day
tod_order  = ["madrugada", "mañana", "tarde", "noche"]
tod_counts = df_full["time_of_day"].value_counts().reindex(tod_order)
axes[1].bar(tod_counts.index, tod_counts.values,
            color=sns.color_palette("muted", 4), alpha=0.85, edgecolor="white")
axes[1].set_title("time_of_day")
axes[1].set_ylabel("Órdenes")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
axes[1].tick_params(axis="x", rotation=20)

# days_bucket (NaN = primera orden)
db_counts = df_full["days_bucket"].value_counts(dropna=False)
axes[2].bar(db_counts.index.astype(str), db_counts.values,
            color=sns.color_palette("coolwarm", len(db_counts)), alpha=0.85, edgecolor="white")
axes[2].set_title("days_bucket")
axes[2].set_ylabel("Órdenes")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
axes[2].tick_params(axis="x", rotation=35)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "etl_03_engineered_features.png", bbox_inches="tight")
plt.show()
print("Guardado: outputs/figures/etl_03_engineered_features.png")

---
## 7. Guardado de Artefactos

In [ ]:
# ── 7.1 Guardar en parquet ────────────────────────────────────────────────────
# Parquet conserva dtypes, es columnar y comprime ~5× vs CSV

artifacts = {
    "master_prior.parquet"       : df_master,
    "df_full.parquet"            : df_full,
    "df_apriori.parquet"         : df_apriori,
    "purchase_frequency.parquet" : df_freq,
}

print("Guardando artefactos en data/processed/:")
for filename, df_art in artifacts.items():
    path = PROCESSED_DIR / filename
    df_art.to_parquet(path, index=False)
    size_mb = path.stat().st_size / 1e6
    print(f"  ✅ {filename:<42} {len(df_art):>12,} filas | {size_mb:.1f} MB")

In [ ]:
# ── 7.2 Resumen final ────────────────────────────────────────────────────────
print("=" * 60)
print("RESUMEN — PIPELINE DE PREPROCESAMIENTO")
print("=" * 60)
print(f"  Filas originales             : {len(df_master):>12,}")
print(f"  Filas post-limpieza (full)   : {len(df_full):>12,}")
print(f"  Retención                    : {len(df_full)/len(df_master)*100:>11.1f}%")
print(f"  Órdenes para CF (ALS)        : {df_full['order_id'].nunique():>12,}")
print(f"  Órdenes para AR (FP-Growth)  : {df_apriori['order_id'].nunique():>12,}")
print(f"  Usuarios únicos              : {df_full['user_id'].nunique():>12,}")
print(f"  Productos únicos             : {df_full['product_id'].nunique():>12,}")
print(f"  Features añadidas            : {[c for c in df_full.columns if c not in df_master.columns]}")
print(f"  Nulos tratados               : days_since_prior_order → days_bucket")
print(f"  Outliers                     : conservados (usuarios activos son valiosos)")
print(f"  Desbalance reordered         : documentado, no corregido (AR/CF insensibles)")
print("=" * 60)
print("\n➡️  Siguiente: notebook 03 — Modelo FP-Growth")